# TuRBO-ENN

This code implements TuRBO [1], a SOTA Bayesian optimization algorithm.

The optimization class, `Turbo`, supports four modes of operation.

**LHD_ONLY**  
Generate a Latin Hypercube Design (LHD) for every batch of arms. This is included as a simple baseline.

**TURBO_ZERO**  
Initialze with LHD. Afterward, sample near the best-so-far x value, x_best. Samples are "near" x_best in two senses: (i) They are in a trust region, an adaptively-sized box around x_best, and (ii) They perturb only a small number of dimensions using RAASP sampling [2]. Other dimensions take the same value as in x_best. The num_arms proposals are chosen randomly from RAASP candidates inside the trust region.

This is included to help differentiate the impact of the trust region from the impact of the surrogate. Notice (below) that the trust region has high impact.

**TURBO_ONE**  
This adds a GP surrogate to TURBO_ZERO. The num_arms proposals are chosen via Thompson sampling from RAASP candidates inside the trust region. Occasionally, the trust region adapter resets and (i) discards all observations, and (ii) begins anew with and LHD design.

This is the standard SOTA method. It should match the TuRBO reference [implementation](https://github.com/uber-research/TuRBO). 

**TURBO_ENN**  
This replaces the GP surrogate with a simpler, more scalable surrogate called Epistemic Nearest Neighbors (ENN). ENN's proposal time scales as $O(N)$ rather than the $O(N^2)$ of a GP surrogate. [3]


## References

1. **Eriksson, D., Pearce, M., Gardner, J. R., Turner, R., & Poloczek, M. (2020).** Scalable Global Optimization via Local Bayesian Optimization. *Advances in Neural Information Processing Systems, 32*.  
   https://arxiv.org/abs/1910.01739

2. **Rashidi, B., Johnstonbaugh, K., & Gao, C. (2024).** Cylindrical Thompson Sampling for High-Dimensional Bayesian Optimization. *Proceedings of The 27th International Conference on Artificial Intelligence and Statistics* (pp. 3502–3510). PMLR.  
   https://proceedings.mlr.press/v238/rashidi24a.html

3. **Sweet, D., & Jadhav, S. A. (2025).** Taking the GP Out of the Loop. *arXiv preprint arXiv:2506.12818*.  
   https://arxiv.org/abs/2506.12818


---

In [ ]:
import numpy as np


class Levy:
    def __init__(self):
        self.bounds = [-10, 10]

    def __call__(self, x):
        x = np.asarray(x, dtype=float)
        num_dim = x.shape[1]
        if x.ndim == 1:
            x = x[None, :]
        x = 10 * x
        w = 1.0 + (x - 1.0) / 4.0
        part1 = np.sin(np.pi * w[:, 0]) ** 2
        num_dim = w.shape[1]
        w_except_last = w[:, : num_dim - 1]
        part2 = np.sum(
            (w_except_last - 1) ** 2
            * (1 + 10 * np.sin(np.pi * w_except_last + 1) ** 2),
            axis=1,
        )
        part3 = (w[:, -1] - 1.0) ** 2 * (1.0 + np.sin(2.0 * np.pi * w[:, -1]) ** 2)
        result = -(part1 + part2 + part3) / num_dim
        return result if result.ndim > 0 else float(result)

In [ ]:
import time

from enn import TurboMode, Turbo


def run_optimization(turbo_mode: TurboMode):
    num_dim = 30
    num_iterations = 300
    num_arms = 3

    objective = Levy()
    bounds = np.array([objective.bounds] * num_dim, dtype=float)

    rng = np.random.default_rng(42)
    optimizer = Turbo(
        bounds=bounds,
        mode=turbo_mode,
        num_arms=num_arms,
        rng=rng,
        k=10,
    )

    best_values = []
    proposal_times = []
    best_y = -np.inf

    for iteration in range(num_iterations):
        t_0 = time.time()
        x_candidates = optimizer.ask(num_arms=num_arms)
        t_1 = time.time()
        proposal_times.append(t_1 - t_0)

        y_values = objective(x_candidates)

        optimizer.tell(x_candidates, y_values)

        current_best = float(np.max(y_values))
        if current_best > best_y:
            best_y = current_best
        best_values.append(best_y)
        if iteration % 30 == 0:
            print(f"Iteration {iteration} best value: {best_y}")

    evals = num_arms * np.arange(len(best_values))
    return best_values, proposal_times, evals

In [ ]:
import matplotlib.pyplot as plt

best_values_zero, proposal_times_zero, evals_zero = run_optimization(
    TurboMode.TURBO_ZERO
)
best_values_one, proposal_times_one, evals_one = run_optimization(TurboMode.TURBO_ONE)
best_values_enn, proposal_times_enn, evals_enn = run_optimization(TurboMode.TURBO_ENN)
best_values_lhd, proposal_times_lhd, evals_lhd = run_optimization(TurboMode.LHD_ONLY)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(evals_zero, best_values_zero, linewidth=2, label="TURBO_ZERO")
plt.plot(evals_enn, best_values_enn, linewidth=2, label="TURBO_ENN")
plt.plot(evals_lhd, best_values_lhd, linewidth=2, label="LHD_ONLY")
plt.plot(evals_one, best_values_one, linewidth=2, label="TURBO_ONE")
plt.xlabel("Function Evaluations")
plt.ylabel("Best Function Value")
plt.title("Convergence Comparison: All Turbo Modes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(evals_zero, proposal_times_zero, linewidth=2, label="TURBO_ZERO")
plt.plot(evals_enn, proposal_times_enn, linewidth=2, label="TURBO_ENN")
plt.plot(evals_lhd, proposal_times_lhd, linewidth=2, label="LHD_ONLY")
plt.plot(evals_one, proposal_times_one, linewidth=2, label="TURBO_ONE")
plt.xlabel("Function Evaluations")
plt.ylabel("Proposal Time (seconds)")
plt.title("Proposal Time vs Function Evaluations: All Turbo Modes")
c = plt.axis()
plt.axis([c[0], c[1], 0, 15])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(evals_zero, proposal_times_zero, linewidth=2, label="TURBO_ZERO")
plt.plot(evals_enn, proposal_times_enn, linewidth=2, label="TURBO_ENN")
plt.plot(evals_lhd, proposal_times_lhd, linewidth=2, label="LHD_ONLY")
plt.xlabel("Function Evaluations")
plt.ylabel("Proposal Time (seconds)")
plt.title("Zoom 100X on Proposal Time (Excludes TURBO_ONE)")
c = plt.axis()
plt.axis([c[0], c[1], 0, 15 / 100])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()